# CREAM Multi-Seed Analysis — AWA2 & CUB
- Section 1: Setup & data loading
- Section 2: Task & concept accuracy (mean ± std across seeds)
- Section 3: Intervention curves (mean ± std across seeds)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

EXPERIMENTS_ROOT = Path('/home/dani00003/mCREAM/experiments')

DATASETS = {
    'AWA2': {
        'model':   'Standard_AWA2_resnet101',
        'exp':     'CREAM_awa2',
        'seeds':   [0, 1, 2],
        'color':   '#2980b9',
    },
    'CUB': {
        'model':   'Standard_CUB',
        'exp':     'CREAM_cub',
        'seeds':   [0, 1, 2, 3, 4],
        'color':   '#27ae60',
    },
}

def seed_dir(dataset, seed):
    cfg = DATASETS[dataset]
    return (EXPERIMENTS_ROOT / dataset / 'train_cbm' / cfg['model']
            / 'CREAM' / cfg['exp'] / f'seed_{seed}')

# Verify paths
for ds, cfg in DATASETS.items():
    for s in cfg['seeds']:
        p = seed_dir(ds, s)
        print(f'{ds} seed_{s}: {"OK" if p.exists() else "NOT FOUND"} — {p.relative_to(EXPERIMENTS_ROOT)}')

---
# Section 1 — Load last_metrics CSVs

In [ ]:
def load_last_metrics(dataset, seed):
    d = seed_dir(dataset, seed)
    csvs = list(d.rglob('last_metrics/*.csv'))
    if not csvs:
        return None
    df = pd.concat([pd.read_csv(f) for f in csvs], ignore_index=True)
    df['seed'] = seed
    df['dataset'] = dataset
    return df

all_metrics = []
for ds in DATASETS:
    for s in DATASETS[ds]['seeds']:
        df = load_last_metrics(ds, s)
        if df is not None:
            all_metrics.append(df)
        else:
            print(f'[MISSING] {ds} seed_{s} — job may still be running')

if all_metrics:
    metrics_df = pd.concat(all_metrics, ignore_index=True)
    print('Loaded metrics shape:', metrics_df.shape)
    print('Columns:', list(metrics_df.columns))
    display(metrics_df.head())
else:
    metrics_df = pd.DataFrame()
    print('No metrics found yet.')

---
# Section 2 — Task & Concept Accuracy (mean ± std across seeds)

In [ ]:
METRIC_COLS = ['test_task_accuracy', 'test_concept_accuracy']

if not metrics_df.empty:
    rows = []
    for ds in DATASETS:
        sub = metrics_df[metrics_df['dataset'] == ds]
        if sub.empty:
            continue
        row = {'dataset': ds, 'n_seeds': sub['seed'].nunique()}
        for m in METRIC_COLS:
            if m in sub.columns:
                row[f'{m}_mean'] = round(sub[m].mean(), 4)
                row[f'{m}_std']  = round(sub[m].std(),  4)
        rows.append(row)
    summary = pd.DataFrame(rows).set_index('dataset')
    print('=== Accuracy Summary ===')
    display(summary)
else:
    print('No metrics loaded.')

In [ ]:
if not metrics_df.empty and 'test_task_accuracy' in metrics_df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle('CREAM — Task & Concept Accuracy across seeds', fontsize=12, fontweight='bold')

    for ax, metric, ylabel in [
        (axes[0], 'test_task_accuracy',    'Task Accuracy'),
        (axes[1], 'test_concept_accuracy', 'Concept Accuracy'),
    ]:
        if metric not in metrics_df.columns:
            ax.set_title(f'{ylabel} — not found'); continue

        datasets = [ds for ds in DATASETS if ds in metrics_df['dataset'].values]
        means = [metrics_df[metrics_df['dataset']==ds][metric].mean() for ds in datasets]
        stds  = [metrics_df[metrics_df['dataset']==ds][metric].std()  for ds in datasets]
        colors = [DATASETS[ds]['color'] for ds in datasets]

        bars = ax.bar(datasets, means, yerr=stds, capsize=6,
                      color=colors, alpha=0.8, width=0.5)
        for bar, m, s in zip(bars, means, stds):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + s + 0.005,
                    f'{m:.3f}±{s:.3f}', ha='center', va='bottom', fontsize=9)
        ax.set_title(ylabel, fontweight='bold')
        ax.set_ylabel(ylabel)
        ax.set_ylim(0, 1.1)

    plt.tight_layout()
    plt.show()

---
# Section 3 — Intervention Curves (mean ± std across seeds)

In [ ]:
def load_interventions(dataset, seed):
    d = seed_dir(dataset, seed)
    csvs = list(d.rglob('intervention_results.csv'))
    if not csvs:
        return None
    df = pd.concat([pd.read_csv(f) for f in csvs], ignore_index=True)
    df['seed'] = seed
    df['dataset'] = dataset
    return df

all_iv = []
for ds in DATASETS:
    for s in DATASETS[ds]['seeds']:
        df = load_interventions(ds, s)
        if df is not None:
            all_iv.append(df)
        else:
            print(f'[MISSING] {ds} seed_{s} intervention_results.csv')

if all_iv:
    iv_df = pd.concat(all_iv, ignore_index=True)
    print('Columns:', list(iv_df.columns))
    display(iv_df.head())
else:
    iv_df = pd.DataFrame()
    print('No intervention results found yet.')

In [ ]:
if not iv_df.empty and 'test_task_accuracy' in iv_df.columns:
    has_group = 'group_interventions' in iv_df.columns
    modes = [False, True] if has_group else [False]
    mode_labels = {False: 'Individual concept interventions', True: 'Group interventions'}

    for group_flag in modes:
        subset = iv_df[iv_df['group_interventions'] == group_flag] if has_group else iv_df
        if subset.empty:
            continue

        datasets_present = [ds for ds in DATASETS if ds in subset['dataset'].values]
        fig, axes = plt.subplots(1, len(datasets_present),
                                 figsize=(7 * len(datasets_present), 5),
                                 sharey=False)
        if len(datasets_present) == 1:
            axes = [axes]
        fig.suptitle(f'CREAM — {mode_labels[group_flag]}',
                     fontsize=12, fontweight='bold')

        for ax, ds in zip(axes, datasets_present):
            color = DATASETS[ds]['color']
            ds_data = subset[subset['dataset'] == ds]
            agg = (ds_data.groupby('num_interventions')['test_task_accuracy']
                   .agg(['mean', 'std', 'count']).reset_index())

            ax.plot(agg['num_interventions'], agg['mean'],
                    lw=2.5, marker='o', markersize=4, color=color,
                    label=f'mean (n={agg["count"].iloc[0]} seeds)')
            ax.fill_between(agg['num_interventions'],
                            agg['mean'] - agg['std'],
                            agg['mean'] + agg['std'],
                            alpha=0.2, color=color)

            # mark 0 and max
            acc0   = agg[agg['num_interventions']==0]['mean'].values
            acc_max = agg.iloc[-1]
            if len(acc0):
                ax.axhline(acc0[0], color='gray', lw=1, ls=':', alpha=0.6, label=f'no interv: {acc0[0]:.3f}')
            ax.scatter(acc_max['num_interventions'], acc_max['mean'],
                       color=color, s=80, zorder=5,
                       label=f'full interv: {acc_max["mean"]:.3f}±{acc_max["std"]:.3f}')

            ax.set_title(ds, fontweight='bold')
            ax.set_xlabel('Number of concepts intervened on')
            ax.set_ylabel('Task Accuracy')
            ax.set_ylim(bottom=0)
            ax.legend(fontsize=8)

        plt.tight_layout()
        plt.show()

    # Summary table
    print('=== Intervention Summary ===')
    rows = []
    for ds in DATASETS:
        for group_flag in ([False, True] if has_group else [False]):
            sub = iv_df[(iv_df['dataset']==ds)]
            if has_group:
                sub = sub[sub['group_interventions']==group_flag]
            if sub.empty: continue
            iv0   = sub[sub['num_interventions']==0]['test_task_accuracy']
            iv_max = sub[sub['num_interventions']==sub['num_interventions'].max()]['test_task_accuracy']
            rows.append({
                'dataset': ds,
                'mode': 'group' if group_flag else 'individual',
                'acc@0_mean': round(iv0.mean(), 4),
                'acc@0_std':  round(iv0.std(),  4),
                'acc@max_mean': round(iv_max.mean(), 4),
                'acc@max_std':  round(iv_max.std(),  4),
            })
    display(pd.DataFrame(rows).set_index(['dataset','mode']))
else:
    print('No intervention data loaded yet.')